# Weather Impact Analysis

How weather conditions affect `arrival_delay`: rain, heavy rain, wind, snow and temperature.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.meteo as an

TRAIN, TEST, lf = setup_analysis("03_analysis_5-meteo")
lf_all   = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])
lf_delay = lf_all.filter(pl.col("canceled") == False)
lf_clean = (
    lf_all
    .filter(pl.col("canceled") == False)
    .filter(~((pl.col("operating_date").dt.year() == 2025) & (pl.col("operating_date").dt.month() >= 11)))
    .filter(pl.col("line_name") != "E")
    .filter(pl.col("stop_sequence") > 1)
)

%load_ext autoreload
%autoreload 2

## Wetterübersicht — alle Faktoren im Vergleich

Vergleich aller binären Wetterbedingungen auf einen Blick: Regen, starker Regen, Wind, Schnee — jeweils True vs. False. Zeigt welcher Faktor den grössten Einfluss hat.

In [ ]:
an.plot_weather_overview(lf_delay, cfg)

In [ ]:
show_df(an.table_weather_overview(lf_delay))

**Beobachtung:** Schnee hat den stärksten Einzeleffekt (+54.0s, OTP −10.9pp), gefolgt von Starkregen (+23.3s). Leichter Regen ist messbar aber moderat (+8.9s).

**`is_windy` — Feature-Idee, aber nicht nutzbar:**
Wind als Feature wurde untersucht, zeigt aber NaN in der gesamten Analyse. Ursache: Das Feature war in der Datenvorbereitung als Feature-Idee vorgesehen, wurde aber nie korrekt befüllt (vermutlich keine Tage mit Wind > 40km/h im Datensatz, oder das Feature wurde nie in die parquet-Dateien geschrieben).
Inhaltlich: Zürich ist durch Bebauung und Hügellagen relativ windgeschützt. Trams sind schwer und auf Schienen gebunden — Wind unter ~60 km/h hat kaum messbaren Betriebseffekt. **`is_windy` wird aus dem Feature-Set entfernt.** (→ F-WEAT-03)

**Wetter-Effekte im Überblick (3 valide Features):**
| Bedingung | Δ Delay (s) | OTP Normal | OTP Wetter | N |
|:---|---:|---:|---:|---:|
| Regen | +8.9 | 87.4% | 84.3% | 10.2M |
| Starkregen | +23.3 | 87.0% | 79.4% | 228k |
| **Schnee** | **+54.0** | **87.1%** | **76.1%** | 273k |

**Wichtige Einschränkung:** Das ist reine Korrelation, keine Kausalität. Alle Wetter-Features haben niedrige Korrelation mit `arrival_delay` (max 0.042). Wetter alleine erklärt wenig Varianz — Wetter-Features bleiben aber als schwache eigenständige Signale im Modell.

**Niederschlagsintensität** zeigt eine klare Dosis-Wirkungs-Beziehung: <2mm=62.6s → >10mm=89.5s. Das ist der stärkste und klarste Wettereffekt im Notebook.

→ Wetter-Flags behalten: `has_snow`, `precipitation`, `has_rain`, `has_heavy_rain`; `is_windy` entfernen; Multikollinearität mit Monat/Saison beachten.

## Temperatur — Kontinuierlicher Effekt

Temperatur in 5°C-Bins: zeigt ob der Effekt linear ist oder ob es Schwellwerte gibt (z.B. Frost unter 0°C).

In [ ]:
an.plot_temperature_precipitation(lf_delay, cfg)

In [ ]:
show_df(an.table_temperature_bins(lf_delay))

**Beobachtung:** Der Temperatureffekt ist monoton ansteigend — **kältere Temperaturen haben WENIGER Delay, wärmere MEHR**.

**Ø Delay nach Temperaturbereich (5°C-Bins):**
| Temperatur | Ø Delay (s) | OTP |
|:---|---:|---:|
| −5–0°C | 54.5 | 88.9% |
| **0–5°C** | **53.8** | **88.1%** (niedrigster Delay!) |
| 5–10°C | 55.4 | 87.4% |
| 15–20°C | 56.7 | 86.8% |
| 25–30°C | 59.7 | 85.3% |
| 35–40°C | 64.0 | 84.6% (n=15k — wenige Daten) |

**Kernbefund:** Die Kälte-Hypothese ist falsch — 0–5°C ist die beste Temperaturzone. Wärme verschlechtert die Pünktlichkeit graduell. Aber der Gesamteffekt ist klein: `is_hot` (>20°C) bringt nur **+2.0s Delta** (55.8s vs. 57.8s, OTP −1.1pp) — im Kontext aller Features ein schwaches Signal.

**Warum mehr Delay bei Wärme?**
- Sommer = mehr Freizeitverkehr, Tourismus, Events → vollere Trams, längere Boardingzeiten
- Gleisausdehnung bei Extremhitze (>30°C) → VBZ-Langsamfahrstellen (klassisches Problem)
- Im 35–40°C-Bin (n=15k) ist der Effekt am stärksten, aber die Datenbasis ist sehr dünn

**Kälte profitiert:** Konsistent mit F-TEMP-06 (Winter = beste Jahreszeit). Mögliche Ursache: weniger MIV bei Schnee/Frost kompensiert Halte-Verzögerungen.

→ `temperature` als kontinuierliches Feature; `is_hot` (>20°C) als binärer Flag; Effekt ist real aber klein (+2s) — nicht überbewerten.

## Feature: `is_hot`

Validierung des `is_hot`-Flags (temperature > 20°C) — binäre Vereinfachung des nicht-linearen Temperatureffekts für das Modell (F-WEAT-04).

In [ ]:
an.plot_is_hot(lf_delay, cfg)

In [ ]:
show_df(an.table_is_hot(lf_delay))

**Beobachtung:** Das `is_hot`-Feature (temperature > 20°C) validiert sich sauber.

**is_hot Vergleich:**
| Kategorie | Ø Delay (s) | OTP | N |
|:---|---:|---:|---:|
| Normal (≤20°C) | ~56s | ~87% | ~66M |
| Heiss (>20°C) | ~58s | ~86% | ~20M |

Der Effekt ist messbar aber moderat — `is_hot` ist ein nützlicher binärer Proxy für den kontinuierlichen Temperatureffekt. Die 20°C-Schwelle trennt zwei klar unterschiedliche Verteilungen, auch wenn der Effekt kleiner ist als der Schnee- oder Starkregen-Effekt.

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status |
|:---|:---|:---|
| F-WEAT-01 | **Schnee** stärkster Wettereffekt: +54.0s, OTP 87.1%→76.1% — klarer Schwellwert-Effekt | done |
| F-WEAT-02 | Starkregen: +23.3s, OTP −7.6pp. Leichter Regen: +8.9s, OTP −3.1pp. Niederschlagsintensität zeigt klare Dosis-Wirkungs-Beziehung (<2mm: 62.6s → >10mm: 89.5s) | done |
| F-WEAT-03 | `is_windy` zeigt NaN — Feature war geplant, wurde nie korrekt befüllt. Inhaltlich kaum relevant (Zürich windgeschützt, Trams schwer). **`is_windy` aus Feature-Set entfernt.** | done |
| F-WEAT-04 | Temperatureffekt: monoton ansteigend. 0–5°C = bester Bereich (53.8s). `is_hot` (>20°C) = +2.0s Delta — schwaches aber reales Signal | done |
| F-WEAT-05 | Alle Wetter-Features haben niedrige Korrelation mit `arrival_delay` (max 0.042). Keine Multikollinearität mit Saison nachweisbar — Wetter und Saison sind weitgehend unabhängige Signale | done |
| F-WEAT-06 | `precipitation` (r=0.036) und `has_snow` (r=0.038) nützlichste Wetter-Features; `temperature` (r=0.018) schwächer; `is_windy` entfernt | done |

## Multikollinearität — Wetter × Saison

In [ ]:
an.plot_multicollinearity_matrix(lf_delay, cfg)

In [ ]:
show_df(an.table_correlation_with_delay(lf_delay))

**Beobachtung:** Die Korrelationsmatrix bestätigt die erwarteten Zusammenhänge.

**Korrelation mit `arrival_delay` (abs. sortiert):**
- `has_snow` hat die stärkste Korrelation (~0.03–0.05) — absolut gering, aber konsistent
- Wetter-Features sind alle schwach korreliert mit Delay (r < 0.1) — Delay ist primär durch betriebliche Faktoren bestimmt
- `season` und `month` korrelieren erwartungsgemäss mit Wetter-Flags (Multikollinearität vorhanden)
- `has_rain` × `season`: negative Korrelation — Sommer (Season=3) ist trockener als Herbst

→ Wetter-Features sind schwache aber valide Prädiktoren; Multikollinearität mit Saison-Features beim Modellbau beachten.